# 🏥 Insurance LLM Fine-Tuning (Colab Edition)

**Model:** Qwen2.5-3B-Instruct + QLoRA (4-bit)  
**GPU:** T4 16GB  
**Training:** SFT → DPO  
**Estimated Time:** ~30-45 min total

---

## Step 0: GPU Check
Make sure **Runtime → Change runtime type → T4 GPU** is selected.

In [ ]:
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## Step 1: Install Dependencies

In [ ]:
%%capture
!pip install -q \
    torch torchvision torchaudio \
    transformers>=4.40.0 \
    datasets>=2.19.0 \
    peft>=0.11.0 \
    trl>=0.8.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.30.0 \
    huggingface-hub>=0.23.0 \
    wandb>=0.17.0 \
    pandas numpy scikit-learn \
    rouge-score nltk evaluate \
    pyyaml python-dotenv tqdm loguru requests

print("✓ Dependencies installed")

## Step 2: Upload Project or Clone from GitHub

**Option A:** Upload your project zip  
**Option B:** Clone from GitHub (if you pushed it)

In [ ]:
# Option A: Upload zip manually
# 1. Zip your D:\insurance-llm-finetuning folder
# 2. Upload to Colab (left sidebar → Files → Upload)
# 3. Uncomment and run:
# !unzip -q /content/insurance-llm-finetuning.zip -d /content/

# Option B: Clone from GitHub
# !git clone https://github.com/YOUR_USERNAME/insurance-llm-finetuning.git

# Option C: Upload from Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/insurance-llm-finetuning /content/

import os
os.chdir('/content/insurance-llm-finetuning')
print(f"Working directory: {os.getcwd()}")
!ls -la

## Step 3: Generate Training Data (if not already done)

In [ ]:
import json
from pathlib import Path

# Check if data already exists
if Path('./data/splits/train.json').exists():
    train_data = json.load(open('./data/splits/train.json'))
    print(f"✓ Data already exists: {len(train_data)} train examples")
else:
    print("Generating data...")
    !python scripts/prepare_data.py --step all --num-examples 1000 --template-ratio 1.0
    train_data = json.load(open('./data/splits/train.json'))
    print(f"✓ Generated: {len(train_data)} train examples")

## Step 4: Load Model + QLoRA Setup

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# QLoRA 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model
print(f"Loading {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare for QLoRA training
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print(f"\n✓ Model loaded")
print(f"  Parameters: {model.num_parameters():,}")
print(f"  GPU memory: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
# Apply LoRA adapters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\n✓ LoRA applied")
print(f"  Trainable: {trainable:,} ({trainable/total*100:.2f}%)")
print(f"  Total: {total:,}")
print(f"  GPU memory: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

## Step 5: Prepare Dataset

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = """You are an expert insurance support agent for a Turkish insurance company.

You help customers with:
- Policy inquiries and explanations
- Claims processing guidance
- Coverage questions
- Premium and billing information
- Policy modifications and renewals

Respond professionally, accurately, and within company policies. Always be helpful and clear."""

def format_chatml(example):
    """Format example to ChatML."""
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}\n<|im_end|>\n"
        f"<|im_start|>user\n{example['user']}\n<|im_end|>\n"
        f"<|im_start|>assistant\n{example['assistant']}\n<|im_end|>"
    )
    return {"text": text}

# Load and format
train_data = json.load(open('./data/splits/train.json'))
val_data = json.load(open('./data/splits/validation.json'))

train_dataset = Dataset.from_list(train_data).map(format_chatml)
val_dataset = Dataset.from_list(val_data).map(format_chatml)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"\nSample ChatML:")
print(train_dataset[0]['text'][:300] + '...')

## Step 6: SFT Training 🚀

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

sft_output_dir = "./outputs/checkpoints/sft"

sft_config = SFTConfig(
    output_dir=sft_output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,      # Effective batch = 2*4 = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    max_seq_length=1024,
    packing=False,
    dataset_text_field="text",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=25,
    report_to="none",
    seed=42,
    run_name="insurance-sft",
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"\n✓ SFT Trainer ready")
print(f"  Epochs: {sft_config.num_train_epochs}")
print(f"  Batch: {sft_config.per_device_train_batch_size} x {sft_config.gradient_accumulation_steps} = {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"  LR: {sft_config.learning_rate}")
print(f"  Max seq: {sft_config.max_seq_length}")

In [ ]:
# Train SFT
print("Starting SFT training...")
sft_result = sft_trainer.train()

print(f"\n✓ SFT Training Complete!")
print(f"  Train loss: {sft_result.metrics.get('train_loss', 'N/A')}")
print(f"  GPU memory: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
# Save SFT adapter
sft_adapter_path = f"{sft_output_dir}/final_adapter"
sft_trainer.save_model(sft_adapter_path)
tokenizer.save_pretrained(sft_adapter_path)
print(f"✓ SFT adapter saved: {sft_adapter_path}")

## Step 7: Generate DPO Preference Data

In [ ]:
import random

REJECTION_STRATEGIES = [
    lambda c: "I'm not sure about that. You should check your policy documents or call us back later.",
    lambda c: "Yes, that should be covered. Let me know if you have other questions.",
    lambda c: "Just send us an email about it and we'll figure it out eventually.",
    lambda c: "Look, I don't know the details of your policy off the top of my head. You'll need to check that yourself.",
    lambda c: "Don't worry, everything is definitely covered under your policy. We'll take care of everything no matter what.",
    lambda c: "Please check your policy.",
    lambda c: "Thank you for your question. Our policies vary and I would recommend reviewing your specific policy documentation for more details.",
    lambda c: "That's handled by a different department. You'll need to call them directly during business hours.",
]

def build_preference_pairs(examples, seed=42):
    rng = random.Random(seed)
    pairs = []
    for ex in examples:
        prompt = (
            f"<|im_start|>system\n{SYSTEM_PROMPT}\n<|im_end|>\n"
            f"<|im_start|>user\n{ex['user']}\n<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        strategy = rng.choice(REJECTION_STRATEGIES)
        pairs.append({
            "prompt": prompt,
            "chosen": ex['assistant'] + "\n<|im_end|>",
            "rejected": strategy(ex['assistant']) + "\n<|im_end|>",
        })
    return pairs

train_pairs = build_preference_pairs(train_data, seed=42)
val_pairs = build_preference_pairs(val_data, seed=43)

train_pref_dataset = Dataset.from_list(train_pairs)
val_pref_dataset = Dataset.from_list(val_pairs)

print(f"✓ Preference pairs: Train={len(train_pref_dataset)} | Val={len(val_pref_dataset)}")

## Step 8: DPO Training 🚀

In [ ]:
from trl import DPOTrainer, DPOConfig

# Clear VRAM from SFT trainer
del sft_trainer
torch.cuda.empty_cache()

dpo_output_dir = "./outputs/checkpoints/dpo"

dpo_config = DPOConfig(
    output_dir=dpo_output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    beta=0.1,
    loss_type="sigmoid",
    max_length=1024,
    max_prompt_length=512,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=25,
    report_to="none",
    seed=42,
    run_name="insurance-dpo",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,  # Uses PEFT implicit reference
    args=dpo_config,
    train_dataset=train_pref_dataset,
    eval_dataset=val_pref_dataset,
    processing_class=tokenizer,
)

print("Starting DPO training...")
dpo_result = dpo_trainer.train()

print(f"\n✓ DPO Training Complete!")
print(f"  Train loss: {dpo_result.metrics.get('train_loss', 'N/A')}")

In [ ]:
# Save DPO adapter
dpo_adapter_path = f"{dpo_output_dir}/final_adapter"
dpo_trainer.save_model(dpo_adapter_path)
tokenizer.save_pretrained(dpo_adapter_path)
print(f"✓ DPO adapter saved: {dpo_adapter_path}")

## Step 9: Quick Evaluation

In [ ]:
# Test inference with fine-tuned model
test_questions = [
    "What is my deductible on my auto insurance policy?",
    "I was in a car accident. How do I file a claim?",
    "Does my policy cover roadside assistance?",
    "Why did my premium increase this year?",
    "I want to add my spouse to my policy.",
]

model.eval()
print("=" * 60)
print("INFERENCE TEST (Fine-tuned Model)")
print("=" * 60)

for q in test_questions:
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}\n<|im_end|>\n"
        f"<|im_start|>user\n{q}\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    print(f"\nQ: {q}")
    print(f"A: {response[:300]}")
    print("-" * 40)

In [ ]:
# Run offline metrics
from src.evaluation.metrics import compute_task_metrics

test_data = json.load(open('./data/splits/test.json'))
references = [ex['assistant'] for ex in test_data]
categories = [ex.get('category', 'unknown') for ex in test_data]

# Generate predictions for test set
predictions = []
for ex in test_data:
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}\n<|im_end|>\n"
        f"<|im_start|>user\n{ex['user']}\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.1, do_sample=True, pad_token_id=tokenizer.pad_token_id)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    predictions.append(response)

# Compute metrics
metrics = compute_task_metrics(predictions, references, categories)

print("\n" + "=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"ROUGE-1: {metrics['overall']['rouge_1']:.4f}")
print(f"ROUGE-2: {metrics['overall']['rouge_2']:.4f}")
print(f"ROUGE-L: {metrics['overall']['rouge_l']:.4f}")
print(f"BLEU:    {metrics['overall']['bleu']:.4f}")
print(f"Keywords:{metrics['overall']['keyword_coverage']:.4f}")
print(f"Format:  {metrics['overall']['format_compliance']:.4f}")

print("\nPer category:")
for cat, m in sorted(metrics['per_category'].items()):
    print(f"  {cat:25s} ROUGE-L:{m['rouge_l']:.4f} KW:{m['keyword_coverage']:.4f}")

# Save results
with open('./outputs/evaluation/colab_results.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"\n✓ Results saved to outputs/evaluation/colab_results.json")

## Step 10: Download Results

In [ ]:
# Zip and download results
!zip -r /content/training_results.zip \
    outputs/checkpoints/sft/final_adapter \
    outputs/checkpoints/dpo/final_adapter \
    outputs/evaluation/ \
    outputs/logs/

from google.colab import files
files.download('/content/training_results.zip')
print("\n✓ Results downloaded!")

In [ ]:
# Optional: Save to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r outputs/ /content/drive/MyDrive/insurance-llm-results/
# print("✓ Saved to Google Drive")

## Summary

| Phase | Model | Method | GPU | Time |
|---|---|---|---|---|
| SFT | Qwen2.5-3B | QLoRA 4-bit | T4 16GB | ~20 min |
| DPO | SFT Model | QLoRA 4-bit | T4 16GB | ~10 min |
| Eval | DPO Model | Inference | T4 16GB | ~5 min |

**Outputs:**
- `outputs/checkpoints/sft/final_adapter/` — SFT LoRA weights
- `outputs/checkpoints/dpo/final_adapter/` — DPO LoRA weights
- `outputs/evaluation/colab_results.json` — Evaluation metrics